# Loopers API on a Colab T4 behind ngrok

Runs `api.main:app` (EmbeddingGemma 2, reranker, ColSmol, faster-whisper on the GPU) and exposes it on a fixed ngrok domain. Point the Vercel `API_URL` at that domain.

**Before you run:** Runtime > Change runtime type > T4 GPU. Add these in the key icon (Secrets) and turn on Notebook access:

- `NGROK_AUTHTOKEN` and `NGROK_DOMAIN` (your free static domain, for example `name.ngrok-free.app`)
- `QDRANT_URL` and `QDRANT_API_KEY` (Qdrant Cloud)
- `DEEPSEEK_API_KEY` (chat) and `HF_TOKEN` (EmbeddingGemma 2 is gated)
- optional: `GROQ_API_KEY`, `GEMINI_API_KEY`, `GITHUB_TOKEN` (only if the repo is private), `FRONTEND_ORIGIN` (for example `https://loopers-psi.vercel.app`)

Upload `deploy_data.zip` (course.db, page images, sources) to Google Drive at `MyDrive/loopers/deploy_data.zip`.

In [ ]:
import os, subprocess
from google.colab import userdata

print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to T4')

def secret(name, required=True):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise SystemExit(f'Missing Colab secret {name} (key icon in the left bar, enable Notebook access)')
        return ''

REQUIRED = ['NGROK_AUTHTOKEN', 'NGROK_DOMAIN', 'QDRANT_URL', 'QDRANT_API_KEY', 'DEEPSEEK_API_KEY', 'HF_TOKEN']
OPTIONAL = ['GROQ_API_KEY', 'GEMINI_API_KEY', 'GITHUB_TOKEN', 'FRONTEND_ORIGIN']
S = {k: secret(k) for k in REQUIRED}
S.update({k: secret(k, required=False) for k in OPTIONAL})
print('secrets ok:', [k for k, v in S.items() if v])

In [ ]:
REPO = 'github.com/Aryan1092raj/Loopers.git'
url = f"https://{S['GITHUB_TOKEN'] + '@' if S['GITHUB_TOKEN'] else ''}{REPO}"
if os.path.isdir('/content/Loopers/.git'):
    !git -C /content/Loopers pull --ff-only
else:
    !git clone --depth 1 {url} /content/Loopers
ROOT = '/content/Loopers/RAG_DB'
print(os.listdir(ROOT)[:12])

In [ ]:
# torch with CUDA already ships with Colab, so it is not reinstalled. The API runs in its own process below,
# so no kernel restart is needed after this.
!pip install -q -r {ROOT}/requirements.app.txt pyngrok

In [ ]:
# Course data: course.db, page images and source files, unzipped into RAG_DB/ (metadata/, processed/, raw/).
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/loopers/deploy_data.zip'
assert os.path.exists(ZIP), f'Upload deploy_data.zip to {ZIP}'
!unzip -q -o {ZIP} -d {ROOT}
print(os.listdir(ROOT + '/metadata')[:8])

In [ ]:
import time, requests

ENV = {
    **os.environ,
    'LOOPERS_NO_REEXEC': '1',
    'MPLBACKEND': 'Agg',
    'QDRANT_URL': S['QDRANT_URL'],
    'QDRANT_API_KEY': S['QDRANT_API_KEY'],
    'COLLECTION': 'multimodal_hackathon',
    'EMBED_MODEL': 'google/embeddinggemma-2',
    'DEEPSEEK_API_KEY': S['DEEPSEEK_API_KEY'],
    'GROQ_API_KEY': S['GROQ_API_KEY'],
    'GEMINI_API_KEY': S['GEMINI_API_KEY'],
    'HF_TOKEN': S['HF_TOKEN'],
    'API_CORS_ORIGINS': ','.join(filter(None, [S['FRONTEND_ORIGIN'], 'http://localhost:3000'])),
}

def start_api():
    log = open('/content/api.log', 'a')
    return subprocess.Popen(
        ['python', '-m', 'uvicorn', 'api.main:app', '--app-dir', 'src', '--host', '0.0.0.0', '--port', '8000'],
        cwd=ROOT, env=ENV, stdout=log, stderr=subprocess.STDOUT,
    )

def wait_healthy(timeout=300):
    end = time.time() + timeout
    while time.time() < end:
        try:
            if requests.get('http://localhost:8000/api/health', timeout=3).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(3)
    return False

api = start_api()
print('api healthy' if wait_healthy() else 'api did not start; last log lines:\n' + ''.join(open('/content/api.log').readlines()[-25:]))

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(S['NGROK_AUTHTOKEN'])
ngrok.kill()
tunnel = ngrok.connect(8000, domain=S['NGROK_DOMAIN'])
PUBLIC = f"https://{S['NGROK_DOMAIN']}"
print('Endpoint:', PUBLIC)
print('Set Vercel API_URL to this, redeploy, and send header ngrok-skip-browser-warning: 1 on proxied calls.')
print(requests.get(PUBLIC + '/api/health', headers={'ngrok-skip-browser-warning': '1'}, timeout=30).text)

In [ ]:
# Keep the session busy and restart the API if it dies. Stop this cell (square button) to shut down.
while True:
    if api.poll() is not None:
        print('api exited, restarting')
        api = start_api()
        wait_healthy()
    time.sleep(30)